In [13]:
import pandas as pd
import geopandas as gpd
import pickle
import numpy as np
import requests

In [14]:
survey_path = "../../../results/surveys/edgt_lyon/trips.parquet"
spatial_path = "../../../results/surveys/edgt_lyon/trips.geoparquet"

calibration_path = "../../../results/road/freeflow/calibration_cache_survey.pickle"
congestion_path = "../../../results/road/congestion_factors.parquet"

# routing_endpoint = "http://sma.univ-eiffel.fr:18054//router/road"
routing_endpoint = "http://localhost:8054/router/road"
departure_time = 4 * 3600
maximum_batch_size = 400

output_path = "../../../results/road/routing.parquet"

In [15]:
# Load survey data
df_survey = pd.read_parquet(survey_path)
df_spatial = gpd.read_parquet(spatial_path)

In [16]:
# Prepare spatial data
df_spatial["origin_x"] = gpd.GeoSeries.from_wkt(df_spatial["origin_geometry"]).x
df_spatial["origin_y"] = gpd.GeoSeries.from_wkt(df_spatial["origin_geometry"]).y
df_spatial["destination_x"] = gpd.GeoSeries.from_wkt(df_spatial["destination_geometry"]).x
df_spatial["destination_y"] = gpd.GeoSeries.from_wkt(df_spatial["destination_geometry"]).y

In [17]:
# Merge in spatial data
df_survey = pd.merge(df_survey, df_spatial)[[
    "trip_id",
    "origin_x", "origin_y",
    "destination_x", "destination_y",
    "departure_time"
]].copy()

In [18]:
# Load freeflow routing parameters
with open(calibration_path, "rb") as f:
    history = pickle.load(f)

objective = np.inf
best = None

for item in history:
    if item["objective"] < objective:
        best = item
        # break # use first

settings = best["settings"]
settings

{'major_factor': np.float64(1.1545020153747612),
 'intermediate_factor': np.float64(0.5000011302575056),
 'minor_factor': np.float64(0.6307488048849574),
 'major_crossing_penalty_s': np.float64(3.20317555754957),
 'equal_crossing_penalty_s': np.float64(3.952579079418678),
 'minor_crossing_penalty_s': np.float64(4.99990825557313)}

In [19]:
# Prepare requests
df_survey["request_index"] = np.arange(len(df_survey))

# Convert to requests
request_list = []

for index, row in df_survey.iterrows():
    request_list.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": departure_time
    })

In [20]:
# Prepare querying
def query_requests(request_list, settings):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(request_list):
        batch = request_list[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size]

        response = requests.post(routing_endpoint, json = {
            "batch": batch,
            "freeflow": settings
        })

        df_response.append(pd.DataFrame.from_records(response.json()))
        batch_index += 1

    return pd.concat(df_response)

In [21]:
# Test connection
assert len(query_requests(request_list[:5], settings)) == 5

In [22]:
# Obtain response
df_response = query_requests(request_list, settings)

In [23]:
df_response = pd.merge(df_response, df_survey[[
    "request_index", "trip_id", "departure_time"
]], on = "request_index")[[
    "trip_id", "departure_time",
    "in_vehicle_distance_km", "in_vehicle_time_min",
    "access_time_min", "egress_time_min",
    "access_distance_km", "egress_distance_km"
]]

In [24]:
# Remove NaN
df_response = df_response[~df_response["departure_time"].isna()]

# Calculate hour
df_response["hour"] = df_response["departure_time"] // 3600
df_response.loc[df_response["hour"] > 23, "hour"] -= 24
df_response["hour"] = df_response["hour"].astype(int)

In [25]:
df_congestion = pd.read_parquet(congestion_path)
df_response = pd.merge(df_response, df_congestion, on = "hour")
df_response["in_vehicle_time_min"] *= df_response["congestion_factor"]

In [26]:
df_response.to_parquet(output_path)